In [ ]:

import pandas as pd

# 加载训练数据和测试数据
train_data_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/reservation_cancellation/train.csv'
test_data_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/reservation_cancellation/test.csv'

train_df = pd.read_csv(train_data_path)
test_df = pd.read_csv(test_data_path)

# 查看训练数据和测试数据的基本信息
print("训练数据基本信息：")
train_df.info()
print("\n训练数据前5行：")
print(train_df.head())

print("\n测试数据基本信息：")
test_df.info()
print("\n测试数据前5行：")
print(test_df.head())


Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

3674 non-null  int64  
 15  no_of_previous_bookings_not_canceled  33674 non-null  int64  
 16  avg_price_per_room                    33674 non-null  float64
 17  no_of_special_requests                33674 non-null  int64  
 18  booking_status                        33674 non-null  int64  
dtypes: float64(1), int64(18)
memory usage: 4.9 MB

训练数据前5行：
      id  no_of_adults  ...  no_of_special_requests  booking_status
0  15559             2  ...                       2               0
1  32783             2  ...                       1               0
2  11797             3  ...                       0               1
3  39750             2  ...                       1               1
4  28711             2  ...                       0               1

[5 rows x 19 columns]

测试数据基本信息：
<class 'pandas.core.frame.DataFrame'>
Ra

In [ ]:


# 检查数据集中是否有缺失值
print("训练数据缺失值统计：")
print(train_df.isnull().sum())

print("\n测试数据缺失值统计：")
print(test_df.isnull().sum())

# 如果没有缺失值，我们可以进入下一步
# 数据预处理通常还包括特征工程，这里我们可以简单地查看数据集的描述统计信息来决定是否进行特征转换
# 这些步骤可能包括标准化、编码分类变量等，但目前来看，所有列都是数值类型，且没有缺失值

# 查看训练数据的描述统计信息
print("\n训练数据描述统计信息：")
print(train_df.describe())

# 根据描述统计信息，决定下一步操作


训练数据缺失值统计：
id                                      0
no_of_adults                            0
no_of_children                          0
no_of_weekend_nights                    0
no_of_week_nights                       0
type_of_meal_plan                       0
required_car_parking_space              0
room_type_reserved                      0
lead_time                               0
arrival_year                            0
arrival_month                           0
arrival_date                            0
market_segment_type                     0
repeated_guest                          0
no_of_previous_cancellations            0
no_of_previous_bookings_not_canceled    0
avg_price_per_room                      0
no_of_special_requests                  0
booking_status                          0
dtype: int64

测试数据缺失值统计：
id                                      0
no_of_adults                            0
no_of_children                          0
no_of_weekend_nights                    

In [ ]:


# 从训练数据中分离目标变量
X_train = train_df.drop(columns=['id', 'booking_status'])
y_train = train_df['booking_status']

# 测试数据中同样去除id列，因为id不是预测所需要的特征
X_test = test_df.drop(columns=['id'])

# 查看分离后的数据
print("X_train前5行：")
print(X_train.head())

print("\ny_train前5行：")
print(y_train.head())

print("\nX_test前5行：")
print(X_test.head())



X_train前5行：
   no_of_adults  no_of_children  ...  avg_price_per_room  no_of_special_requests
0             2               0  ...                60.0                       2
1             2               0  ...               111.6                       1
2             3               0  ...               168.3                       0
3             2               0  ...               115.0                       1
4             2               0  ...               132.3                       0

[5 rows x 17 columns]

y_train前5行：
0    0
1    0
2    1
3    1
4    1
Name: booking_status, dtype: int64

X_test前5行：
   no_of_adults  no_of_children  ...  no_of_special_requests  booking_status
0             2               0  ...                       1               0
1             2               0  ...                       0               1
2             2               0  ...                       0               0
3             2               1  ...                       3               0

In [ ]:



# 从X_test中移除booking_status列
X_test = X_test.drop(columns=['booking_status'])

# 检查移除后的X_test
print("移除booking_status后的X_test前5行：")
print(X_test.head())

# 准备训练LightGBM模型
import lightgbm as lgb

# 创建LightGBM数据集
train_data = lgb.Dataset(X_train, label=y_train)

# 设置LightGBM参数
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': 0
}

# 训练模型
num_rounds = 100
lgb_model = lgb.train(params, train_data, num_rounds)

# 对测试数据进行预测
y_pred = lgb_model.predict(X_test)

# 查看预测结果的前几个值
print("\n预测结果的前5个值：")
print(y_pred[:5])




移除booking_status后的X_test前5行：
   no_of_adults  no_of_children  ...  avg_price_per_room  no_of_special_requests
0             2               0  ...               80.67                       1
1             2               0  ...               93.96                       0
2             2               0  ...              104.65                       0
3             2               1  ...              124.67                       3
4             2               0  ...               90.95                       1

[5 rows x 17 columns]

预测结果的前5个值：
[0.02704412 0.88032027 0.46321216 0.02054472 0.17884121]


In [ ]:



# 加载测试数据集的目标变量
y_test = test_df['booking_status']

# 计算AUC-ROC分数
from sklearn.metrics import roc_auc_score

auc_roc = roc_auc_score(y_test, y_pred)
print(f"AUC-ROC score on the test set: {auc_roc:.4f}")

# 将预测结果保存到文件中
import pandas as pd

# 创建包含预测结果的DataFrame
predictions_df = pd.DataFrame({
    'id': test_df['id'],
    'booking_status_pred': y_pred
})

# 保存预测结果到CSV文件
predictions_df.to_csv('reservation_cancellation_predictions.csv', index=False)

print("预测结果已保存到文件 'reservation_cancellation_predictions.csv'")


AUC-ROC score on the test set: 0.8936
预测结果已保存到文件 'reservation_cancellation_predictions.csv'
